# CribWatch: reproduce the uploaded-dataset research
**The two implemented baselines did not meet 95% accuracy and are not suitable for a baby safety alarm.** See `DATASET_RESULTS.md` in the updated source/research bundle. This notebook reproduces COCO intake, duplicate audit, 12-feature extraction, two exploratory models, and a review queue. It does not infer airway clearance, sleep/awake state, or SIDS outcomes.

Use a fresh CPU Colab runtime. Download the source/research ZIP supplied with this project and upload it in the first cell. Keep infant data, identities and reports private. The project repository includes source and final research artifacts through Git/Git LFS. Use the **original Roboflow ZIPs**, not chat-transfer parts, when uploading the datasets below. The originals stay unchanged.

In [ ]:
import os, sys, subprocess, pathlib, zipfile, stat
PROJECT = pathlib.Path('/content/SeniorDesignProject')
if not PROJECT.is_dir():
    from google.colab import files
    print('Upload cribwatch-source.zip (the source bundle, not a dataset).')
    uploaded = files.upload()
    names = [n for n in uploaded if n.endswith('.zip')]
    if len(names) != 1:
        raise ValueError('Upload exactly one source ZIP')
    with zipfile.ZipFile(names[0]) as archive:
        members = archive.infolist()
        base = pathlib.Path('/content').resolve()
        if len(members) > 1000 or sum(m.file_size for m in members) > 50_000_000:
            raise ValueError('Unexpected source archive size')
        for member in members:
            target = (base / member.filename).resolve()
            if not target.is_relative_to(PROJECT) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('Unexpected source archive path')
            if target.exists():
                raise FileExistsError('Preserving existing files; use a fresh runtime')
        archive.extractall(base)
    del uploaded  # release upload bytes
if not (PROJECT / 'pyproject.toml').is_file():
    raise ValueError('Source bundle does not contain the project')
os.chdir(PROJECT)
# Colab often preinstalls multiple competing cv2 distributions. Keep one provider.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'opencv-python', 'opencv-python-headless',
                'opencv-contrib-python-headless', 'opencv-contrib-python'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT) + '[vision,notebook]'], check=True)
print('If Colab requests a runtime restart after dependency changes, restart and continue below.')

## Initialize and verify model assets
Official versioned pose, face and MobileNetV3 files are verified against pinned SHA256 hashes. The image model uses a frozen generic embedding and a trained small classifier, not an infant-specific fine-tuned network. If Colab requests a restart after installation, restart and continue here.

In [ ]:
import os, sys, subprocess, pathlib, json
import numpy as np
PROJECT = pathlib.Path('/content/SeniorDesignProject')
os.chdir(PROJECT)
from cribwatch.assets import download_models
from cribwatch.vision import Vision
from cribwatch.data import read_csv, review_widget, extract_manifest
from cribwatch.coco import audit_coco
from cribwatch.learning import train
from cribwatch.image_model import extract_embeddings, train_image_baseline, propose_unlabeled
download_models('models')
with Vision('models') as vision:
    assert vision.observe(np.zeros((480,640,3), dtype=np.uint8)).reason == 'no_pose'
print('Real CPU model loading/inference passed; this is not an infant accuracy test.')

## Upload the two original dataset exports
Upload `baby.v5-dataset70_20_10.coco.zip` and `baby-detection.v16i.coco.zip`. They may exceed 32 MiB here: that earlier limit applies to transfers into the chat execution machine, not this Colab upload. For large files you can mount a private Drive folder instead. Colab storage is temporary; retain your reviewed manifests/reports privately.

In [ ]:
from google.colab import files
from cribwatch.assets import safe_unzip
DATA = pathlib.Path('/content/cribwatch-research-data')
DATA.mkdir(exist_ok=True)
EXPORTS = DATA / 'exports'
EXPORTS.mkdir(exist_ok=True)
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if not archives:
    raise ValueError('Select the original Roboflow dataset ZIPs')
ROOTS = {}
for i, name in enumerate(archives):
    target = EXPORTS / f'export-{i}'
    safe_unzip(name, target)
    ROOTS[f'export-{i}'] = target
    if not list(target.rglob('_annotations.coco.json')):
        raise ValueError('No COCO annotations: use original exports, not the numbered transfer parts')
del uploaded

## Audit existing annotations and leakage
`supine` maps to `back`. `baby` and `baby_face` are never treated as posture labels. Publisher labels keep their provenance and are not marked human-verified. Exact-image/same-source annotation conflicts are quarantined. Similar frames stay in the same evaluation group but can legitimately have different postures.

Filename words are not posture ground truth. Source-image groups are not baby/session IDs. Neither supplied export identifies individual babies/recordings, so this route is **exploratory**. Do not invent identities to bypass the independent training path.

In [ ]:
AUDIT = DATA / 'audit'
summary = audit_coco(ROOTS, AUDIT)
print(json.dumps(summary, indent=2))

## Extract the requested 12 metrics
Full-frame CPU inference is used. No ground-truth bounding boxes are used as an oracle crop. Images are stretched to 640x640 to match the supplied posture export; the same preprocessing is applied at inference. Metrics are projected geometry, not physical safety measurements. `head_lift_ratio` is only an estimated camera-depth proxy. Missing/poor landmarks become unknown, not a back prediction.

Extraction checkpoints permit recovery after interruption if input files/manifest/model/schema are unchanged. Keep the checkpoint with its output. Use fresh output paths for changed inputs. Worker count 1 is conservative for Colab.

In [ ]:
FEATURES_CSV = AUDIT / 'features.csv'
print('Images processed:', extract_manifest(AUDIT / 'images.csv', FEATURES_CSV, 'models', workers=1))
rows = read_csv(FEATURES_CSV)
print('Usable torso observations:', sum(r['pose_ok']=='1' for r in rows), '/', len(rows))

## Metrics-only exploratory model
This uses existing publisher posture annotations, with duplicate/source groups separated across train/validation/test. Unknown labels are excluded from supervised ground truth; labeled failures remain in evaluation as errors. Validation chooses the rejection threshold with a requested 50% coverage floor; the report explicitly says when it cannot be met. Test outputs are exploratory comparisons, not independent unseen-baby validation.

In [ ]:
METRIC_RUN = DATA / 'metrics-run'
metric_report = train(FEATURES_CSV, METRIC_RUN, exploratory=True)
print(json.dumps(metric_report, indent=2))
print('NOT a deployment approval. Read overall accuracy AND coverage, not accepted-only accuracy alone.')

## Image + metrics exploratory baseline
This additional comparison uses generic MobileNetV3 embeddings plus the 12 metrics, so it can attempt posture classification when landmarks are absent. It still cannot verify infant presence, a clear airway, or clinical safety. Candidate classifier heads are selected on validation data. No GPU or neural-network fine-tuning is used. Do not tune repeatedly against observed test results and then present the same cohort as independent validation.

In [ ]:
EMBEDDINGS = AUDIT / 'embeddings.npz'
print('Images embedded:', extract_embeddings(FEATURES_CSV, 'models', EMBEDDINGS))
IMAGE_RUN = DATA / 'image-run'
image_report = train_image_baseline(FEATURES_CSV, EMBEDDINGS, IMAGE_RUN)
print(json.dumps(image_report, indent=2))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, title, report in zip(axes, ['12 metrics', 'Image + metrics'], [metric_report, image_report]):
    matrix = np.array(report['confusion_matrix'])[:3]
    ax.imshow(matrix, cmap='Blues')
    ax.set(xticks=range(4), yticks=range(3), xticklabels=['back','side','prone','unknown'],
           yticklabels=['back','side','prone'], xlabel='Predicted', ylabel='Publisher label', title=title)
    for i in range(3):
        for j in range(4):
            ax.text(j, i, str(matrix[i,j]), ha='center', va='center')
plt.tight_layout()
plt.show()

## Propose labels for unknown images; preserve uncertainty
This queue includes the baby/face dataset, which has no posture ground truth. Proposed labels are not accepted automatically: `posture` remains unknown and review flags remain 0. Never use model-generated labels to score the model that generated them. Final test annotations need independent blinded review.

The optional widget lets a person confirm posture and face appearance, with or without identity metadata. Unknown identities still block the standard baby-independent training workflow. Conflicting declared-source labels require adjudication across all copies, not just a one-row edit.

In [ ]:
REVIEW_QUEUE = AUDIT / 'review_queue.csv'
print(propose_unlabeled(FEATURES_CSV, EMBEDDINGS, IMAGE_RUN / 'posture.joblib', REVIEW_QUEUE))
# Uncomment to review visually; do not count proposals as verified labels.
# review_widget(REVIEW_QUEUE, batch_size=12)

## Training distributions are not safety ranges
The metrics-only run writes per-class 5th/50th/95th percentiles from training poses. Inspect overlap. These are camera-dependent descriptions, not universal safe/unsafe cutoffs. No range in this notebook establishes whether an airway is clear.

In [ ]:
print((METRIC_RUN / 'training_feature_percentiles.json').read_text())

## Analyze an authorized recorded video offline
Select one video obtained with permission. Do not place an infant in a risky situation to generate test data. Offline analysis uses video timestamps and never sends a physical alarm. `--allow-exploratory` is available for offline analysis only; live monitoring rejects these artifacts. A flat/blank frame becomes unknown, but patterned occlusions and other out-of-distribution inputs are not reliably detected.

In [ ]:
from cribwatch.video import analyze_video
uploaded = files.upload()
videos = [name for name in uploaded if name.lower().endswith(('.mp4','.mov','.avi','.mkv'))]
if len(videos) != 1:
    raise ValueError('Upload exactly one authorized video')
print('Sampled frames:', analyze_video(videos[0], IMAGE_RUN/'posture.joblib', 'models',
                                     DATA/'video.jsonl', sample_fps=5, allow_exploratory=True))
del uploaded
print('State transitions are not event accuracy. Independent video labels are needed for sensitivity, latency and false alarms/hour.')

## Retain results and next development work
Save models/reports/manifests privately; do not publish infant images or identifiers casually. Joblib is executable Python: load only artifacts you trust. The public-dataset labels still need review, and the project needs identified camera-specific infant/session recordings plus an infant-specific/fine-tuned model and an independent test cohort. Hardware acceptance, alarm behavior, latency/thermal performance and receiver power-loss indication remain separate checks in `deploy/raspberry-pi.md`.

Downloading an entire folder of raw images as one ZIP recreates the large-file problem for chat transfers. You can download small model/report files individually through Colab's Files sidebar. Upload original large datasets to this notebook; use the supplied splitting script only when transferring them into chat.

In [ ]:
for run in [METRIC_RUN, IMAGE_RUN]:
    (run/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))
print('Retain research files from:', DATA)
print('Neither research artifact is approved for live infant monitoring.')